### Showcasing Tool and Memory in LangGraph

In [1]:
from typing import Annotated
from langgraph.graph import StateGraph, START, END
from langgraph.graph.message import add_messages
from IPython.display import Image, display
import gradio as gr
from langgraph.prebuilt import ToolNode, tools_condition
import requests
import os
from langchain_openai import ChatOpenAI
from typing import TypedDict
from pydantic import BaseModel, Field
from config import Config
from langchain_community.utilities import GoogleSerperAPIWrapper
from langchain.tools import tool
from dotenv import load_dotenv


config = Config()
load_dotenv()


/Users/rodrigo/Desktop/work_projects/AI_training/.venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


True

In [2]:
llm = config.get_llm()

In [3]:
@tool
def convert_amount(amount, rate):
    """tool to convert an amount of one currency to Euros
    Args:
        amount (float): the amount to convert
        rate (float): the conversion rate to Euros
    Returns:
        str: the converted amount in Euros"""

    return f"{amount * rate} EUR"

@tool
def search_web(query):
    """tool to search the web for a query when you need updated information 
    that the model does not have access to
    Args:
        query (str): the query to search for
    Returns:
        str: the search results"""
    
    search = GoogleSerperAPIWrapper()
    return search.run(query)

tools = [convert_amount, search_web]

In [4]:
llm_with_tools = llm.bind_tools(tools)

In [5]:
class State(BaseModel):
    """State of the agent, including the conversation history and any relevant information."""
    messages : Annotated[list, add_messages]

In [6]:
app = StateGraph(State)

In [7]:
def chatbot(state: State):
    return {"messages": [llm_with_tools.invoke(state.messages)]}

In [8]:
app.add_node("chatbot", chatbot)
app.add_node("tools", ToolNode(tools=tools))

app.add_edge(START, "chatbot")
app.add_conditional_edges("chatbot", tools_condition, "tools")
app.add_edge("tools", "chatbot")
app.add_edge("chatbot", END)

In [9]:
graph  = app.compile()

In [10]:
def chat(query, history):
    messages = [{"role": "user", "content": query}]
    result = graph.invoke({"messages": messages})
    return result["messages"][-1].content

In [13]:
messages = [{"role": "user", "content": "hey"}]

result = graph.invoke({"messages": messages})


In [14]:
result

{'messages': [HumanMessage(content='hey', additional_kwargs={}, response_metadata={}, id='9c255da1-5694-4723-9599-703a3fcb197a'),
  AIMessage(content='Hello! How can I assist you today?', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 11, 'prompt_tokens': 147, 'total_tokens': 158, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}, 'latency_checkpoint': {'engine_tbt_ms': 5, 'engine_ttft_ms': 71, 'engine_ttlt_ms': 124, 'pre_inference_ms': 174, 'service_tbt_ms': 5, 'service_ttft_ms': 526, 'service_ttlt_ms': 585, 'total_duration_ms': 411, 'user_visible_ttft_ms': 352}}, 'model_provider': 'openai', 'model_name': 'gpt-4o-2024-11-20', 'system_fingerprint': 'fp_af7f7349a4', 'id': 'chatcmpl-DcWQ0ON4GKYHGuGgoYT5fAQG6DPBg', 'service_tier': 'default', 'prompt_filter_results': [{'prompt_index': 0, 'cont

In [11]:
gr.ChatInterface(fn=chat).launch()

* Running on local URL:  http://127.0.0.1:7860
* To create a public link, set `share=True` in `launch()`.
